In [1]:
import sys
sys.path.append("/aloy/home/ddalton/projects/scGPT_playground")
from src.preprocessing import pipeline as pp
from src.utils import utils, utils as ut
import scanpy as sc

manual_parameters = { 
    "dataset_exercise":"umls_clean",                 
    "diseases_of_interest_set": None,
    "library_strategies_of_interest_set": ["RNA-Seq"],
    "normalize": "log2CPM",  # options: "log2CPM", "CPM", None
}

# get dsaids of interest
dsaids_interest = ut.get_doids_with_umls(manual_parameters.get("library_strategies_of_interest_set"))

# get data info
df_info = ut.load_dsa_info()
print(f"Loaded DSAIDs info: {len(df_info)}")

df_filtered = df_info[df_info["dsaid"].isin(dsaids_interest)]
print(f"Filtered DSAIDs info: {len(df_filtered)}")

/aloy/home/ddalton/miniconda3/envs/scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:root:Loading signatures from file /aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/signatures.pkl


Loaded Nº DSAIDs: 10306
Filter Human Species Nº DSAIDs: 7194
Filter ['RNA-Seq'] Nº DSAIDs: 2536
Filter DSAIDs with info Nº DSAIDs: 2534
Loaded signatures Nº DSAIDs: 2534


INFO:root:Nº of Human protein coding genes: 20608


Loaded human protein coding genes: 20608
Filter to enough Human protein coding genes Nº DSAIDs: 2533
Filter by clean DSAIDs Nº DSAIDs: 2533
Filter by presence of UMLS codes Nº DSAIDs: 2421
Filter by mapping from UMLS to DOID Nº DSAIDs: 1607
Loaded DSAIDs info: 10306
Filtered DSAIDs info: 1607


In [2]:
df_filtered["dsaid"].to_list()

['DSA00004',
 'DSA00006',
 'DSA00015',
 'DSA00016',
 'DSA00030',
 'DSA00045',
 'DSA00046',
 'DSA00047',
 'DSA00048',
 'DSA00068',
 'DSA00069',
 'DSA00071',
 'DSA00072',
 'DSA00073',
 'DSA00074',
 'DSA00104',
 'DSA00105',
 'DSA00111',
 'DSA00112',
 'DSA00113',
 'DSA00115',
 'DSA00116',
 'DSA00117',
 'DSA00118',
 'DSA00131',
 'DSA00132',
 'DSA00135',
 'DSA00147',
 'DSA00148',
 'DSA00151',
 'DSA00175',
 'DSA00176',
 'DSA00180',
 'DSA00181',
 'DSA00187',
 'DSA00188',
 'DSA00197',
 'DSA00205',
 'DSA00215',
 'DSA00216',
 'DSA00217',
 'DSA00219',
 'DSA00220',
 'DSA00221',
 'DSA00244',
 'DSA00247',
 'DSA00248',
 'DSA00249',
 'DSA00250',
 'DSA00252',
 'DSA00253',
 'DSA00275',
 'DSA00276',
 'DSA00277',
 'DSA00278',
 'DSA00279',
 'DSA00282',
 'DSA00283',
 'DSA00288',
 'DSA00295',
 'DSA00312',
 'DSA00313',
 'DSA00342',
 'DSA00343',
 'DSA00353',
 'DSA00360',
 'DSA00365',
 'DSA00366',
 'DSA00367',
 'DSA00368',
 'DSA00395',
 'DSA00413',
 'DSA00414',
 'DSA00415',
 'DSA00416',
 'DSA00417',
 'DSA00418',

In [3]:
import scanpy as sc
adata = sc.read_h5ad("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-13-01/data.h5ad")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-13-01/data.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
adata.shape

(0, 20608)

In [ ]:
import importlib
importlib.reload(pp)

<module 'src.preprocessing.pipeline' from '/aloy/home/ddalton/projects/scGPT_playground/src/preprocessing/pipeline.py'>

In [ ]:
df_filtered = df_filtered.copy()
df_filtered["celltype"] = df_filtered["diseaseid"]
df_filtered["ids"] = df_filtered["dsaid"]
df_filtered["dataset_id"] = df_filtered["accession"]


In [ ]:
# quality control
# dsaids >= n_samples
counts_cond = df_filtered["control_case_sample_count"].to_list()
df_filtered = df_filtered.copy()
df_filtered["n_control"] = [int(x.split("|")[0]) for x in counts_cond]
df_filtered["n_cases"] = [int(x.split("|")[1]) for x in counts_cond]
df_filtered["celltype"] = df_filtered["diseaseid"]

for i in [1,2, 5, 10]:
    print(f"Nº DSAIDs with +{i} cases: {len(df_filtered[df_filtered['n_cases'] >= i])}")

for i in [1,2, 5, 10]:
    print(f"Nº DSAIDs with +{i} controls: {len(df_filtered[df_filtered['n_control'] >= i])}")

for i in [1,2, 5, 10]:
    print(f"Nº DSAIDs with +{i} controls & conditions: {len(df_filtered[(df_filtered['n_cases']>= i) & (df_filtered['n_control'] >= i)])}")


df_filtered_2 = df_filtered[(df_filtered['n_cases']>= 2) & (df_filtered['n_control'] >= 2)]
print(f"Filtered DSAIDs info with +2 cases & controls: {len(df_filtered_2)}")

Nº DSAIDs with +1 cases: 1607
Nº DSAIDs with +2 cases: 1506
Nº DSAIDs with +5 cases: 870
Nº DSAIDs with +10 cases: 462
Nº DSAIDs with +1 controls: 1607
Nº DSAIDs with +2 controls: 1511
Nº DSAIDs with +5 controls: 872
Nº DSAIDs with +10 controls: 434
Nº DSAIDs with +1 controls & conditions: 1607
Nº DSAIDs with +2 controls & conditions: 1467
Nº DSAIDs with +5 controls & conditions: 731
Nº DSAIDs with +10 controls & conditions: 326
Filtered DSAIDs info with +2 cases & controls: 1467


In [ ]:
# diseases >= n_datasets
for i in [1,2,3, 5, 10]:
    _n_diseases = sum(df_filtered.groupby('diseaseid')['accession'].nunique()>=i)
    print(f"Nº diseases with +{i} datasets: {_n_diseases}")
v

for i in [1,2,3, 5, 10]:
    _n_diseases = sum(df_filtered_2.groupby('diseaseid')['accession'].nunique()>=i)
    print(f"Nº diseases - pre-filtered -  with +{i} datasets: {_n_diseases}")


disease_ids_interest = [k for k, v in dict(df_filtered_2.groupby('diseaseid')['accession'].nunique()).items() if v>=3 ]

df_filtered_3 = df_filtered_2[df_filtered_2["diseaseid"].isin(disease_ids_interest)]
print(f"Nº DSAIDs with +3 datasets: {len(df_filtered_3)}")

Nº diseases with +1 datasets: 340
Nº diseases with +2 datasets: 141
Nº diseases with +3 datasets: 87
Nº diseases with +5 datasets: 46
Nº diseases with +10 datasets: 17
Nº diseases - pre-filtered -  with +1 datasets: 324
Nº diseases - pre-filtered -  with +2 datasets: 133
Nº diseases - pre-filtered -  with +3 datasets: 82
Nº diseases - pre-filtered -  with +5 datasets: 44
Nº diseases - pre-filtered -  with +10 datasets: 15
Nº DSAIDs with +3 datasets: 999


In [ ]:

def do_you_fuck():
    print(fuck)


fuck = "Yes"
do_you_fuck()

Yes


In [ ]:
import importlib
importlib.reload(pp)

fuck="YES"
pp.do_you_fuck()

NameError: name 'fuck' is not defined

In [ ]:
from src.training import train as tr

In [ ]:
df_filtered = pp.clean_dsaids_qc(df_filtered)

Nº dsaids: 1181	Nº unique diseases: 133
Filter Datasets w/ Samples +2	Nº dsaids: 1181	Nº unique diseases: 133
Filter Diseases w/ Datasets +2	Nº dsaids: 1181	Nº unique diseases: 133


In [ ]:
df_split_2 = tr.split_stratified(df_filtered)

tr.is_test_good(df_split_2)

True

In [ ]:
import pandas as pd




In [ ]:
report_split(df_split_2)

Nº of diseases in train split 1:	133
Nº of diseases in test split 1:	133
Nº of datasets in train split 1:	533
Nº of datasets in test split 1:	135
Nº of samples in train split 1:	942
Nº of samples in test split 1:	239


In [ ]:
df_split_2[df_split_2["accession"] == "GSE174302"]

In [ ]:
df_split_2

,dsaid,accession,platform,deg_count,disease,diseaseid,tissue,data_source,library_strategy,organism,control_case_sample_count,definition,n_control,n_cases,celltype,ids,dataset_id,test_split_1
3,DSA00004,GSE224022,GPL16791,1000,Retinoblastoma,C0035335,Retina,GEO,RNA-Seq,Homo sapiens,4|5,DO:A retinal cell cancer and malignant neoplas...,4,5,C0035335,DSA00004,GSE224022,0
5,DSA00006,GSE126342,GPL11154,1000,Myotonic Dystrophy Type 1,C0027126,Skeletal muscle,GEO,RNA-Seq,Homo sapiens,6|16,DO:A myotonic disease that is characterized by...,6,16,C0027126,DSA00006,GSE126342,1
14,DSA00015,GSE224056,GPL24676,1000,Gastric Cancer,C0699791,Stomach,GEO,RNA-Seq,Homo sapiens,4|5,DO:A gastrointestinal system cancer that is lo...,4,5,C0699791,DSA00015,GSE224056,1
15,DSA00016,GSE224056,GPL24676,112,Gastric Cancer,C0699791,Stomach,GEO,RNA-Seq,Homo sapiens,5|5,DO:A gastrointestinal system cancer that is lo...,5,5,C0699791,DSA00016,GSE224056,1
29,DSA00030,GSE215424,GPL24676,1000,Amyotrophic Lateral Sclerosis,C0002736,Muscle,GEO,RNA-Seq,Homo sapiens,5|5,DO:A motor neuron disease that is characterize...,5,5,C0002736,DSA00030,GSE215424,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10101,DSA10102,GSE163908,GPL24676,935,Idiopathic Pulmonary Fibrosis,C1800706,NaN,GEO,RNA-Seq,Homo sapiens,3|3,DO:A pulmonary fibrosis that is characterized ...,3,3,C1800706,DSA10102,GSE163908,0
10120,DSA10121,GSE29006,GPL10999,0,Lung Cancer,C0684249,Lung,GEO,RNA-Seq,Homo sapiens,2|2,DO:A respiratory system cancer that is located...,2,2,C0684249,DSA10121,GSE29006,0
10130,DSA10131,GSE164264,GPL23227,1000,Acquired Immunodeficiency Syndrome,C0001175,Blood,GEO,RNA-Seq,Homo sapiens,2|2,DO:A viral infectious disease that results in ...,2,2,C0001175,DSA10131,GSE164264,0
10131,DSA10132,GSE164264,GPL23227,1000,Acquired Immunodeficiency Syndrome,C0001175,Blood,GEO,RNA-Seq,Homo sapiens,2|2,DO:A viral infectious disease that results in ...,2,2,C0001175,DSA10132,GSE164264,0


In [ ]:
group_label = "dataset_id"

test_dt = df_split_2[df_split_2[f"test_split_{1}"]==1][group_label].unique()
train_dt = df_split_2[df_split_2[f"test_split_{1}"]==0][group_label].unique()

set(test_dt) & set(train_dt)


set()

In [ ]:
from typing import *
import pandas as pd
import logging
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

import random
population = [1,2,3,4,5]
s_size = (len(population) //10)



def get_stratified_test_split(obs: pd.DataFrame, n_splits=3) -> List[str]:
    """Get Test Split
    We will perform a split for those diseases which have more than one dataset.

    There MUST not be any data-leakage between the train and test set - no shared datasets between the two sets.

    Strategy:
        1. Check diseases w/ n_splits+ datasets
        2. Divide dataset into train and test w/ 10:1 ratio
        3. Assign train and test to the respective datasets

    """

    obs_copy = obs.copy(deep=True)

    # pre-process data
    obs_copy["combination"] = (
        obs_copy["celltype"].astype(str) + "_" + obs_copy["dataset_id"].astype(str)
    )

    logging.info(f"Columns in obs_copy {obs_copy.columns}")

    assert "combination" in obs_copy.columns, "combination column not created in obs"

    diseases_f1 = set()  # diseases filter 1

    # 1. Check diseases w/ n_splits+ datasets
    all_diseases = obs_copy["celltype"].unique()
    for diseases in all_diseases:
        QUERY = f'celltype == "{diseases}"'
        _df_query = obs_copy.query(QUERY)
        if len(_df_query["dataset_id"].unique()) >= n_splits:
            diseases_f1.add(diseases)

    logging.info(f"Number of diseases with {n_splits}+ datasets: {len(diseases_f1)}")

    # 2. Divide dataset into train and test to n_splits
    QUERY = "celltype in @diseases_f1"
    df_diseases_f1 = obs_copy.query(QUERY)

    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    for i, (train_idx, test_idx) in enumerate(
        sgkf.split(
            X=df_diseases_f1["ids"],
            y=df_diseases_f1["celltype"],
            groups=df_diseases_f1["dataset_id"],
        )
    ):

        # get which disease & datasets are in the test
        df_diseases_f1_test = df_diseases_f1.iloc[test_idx]
        df_diseases_f1_train = df_diseases_f1.iloc[train_idx]

        assert "combination" in obs_copy.columns, "combination column not found in obs"
        assert (
            "combination" in df_diseases_f1_test.columns
        ), "combination column not found in df_diseases_f1_test"

        # 3. Assign train and test labels
        obs_copy[f"test_split_{i+1}"] = (
            obs_copy["combination"].isin(df_diseases_f1_test["combination"]).astype(int)
        )

        logging.info(
            f"Nº of diseases in train split {i+1}: {obs_copy.iloc[train_idx]['celltype'].nunique()}"
        )

        logging.info(
            f"Nº of diseases in test split {i+1}: {obs_copy.iloc[test_idx]['celltype'].nunique()}"
        )

        logging.info(
            f"Nº of datasets in train split {i+1}: {obs_copy.iloc[train_idx]['dataset_id'].nunique()}"
        )

        logging.info(
            f"Nº of datasets in test split {i+1}: {obs_copy.iloc[test_idx]['dataset_id'].nunique()}"
        )

        logging.info(
            f"Nº of samples in train split {i+1}: {obs_copy.iloc[train_idx]['ids'].nunique()}"
        )

        logging.info(
            f"Nº of samples in test split {i+1}: {obs_copy.iloc[test_idx]['ids'].nunique()}"
        )

        # assert there is no overlap
        train_dataset_ids = set(df_diseases_f1_train["dataset_id"])
        test_dataset_ids = set(df_diseases_f1_test["dataset_id"])

        overlap = train_dataset_ids.intersection(test_dataset_ids)
 

        assert len(overlap) == 0, f"Overlap found in dataset_ids between train and test in split {i+1}: {overlap}"


    obs_copy.drop(columns=["combination"], inplace=True)

    return obs_copy

df_split = get_stratified_test_split(df_filtered_3, n_splits=3)


INFO:root:Columns in obs_copy Index(['dsaid', 'accession', 'platform', 'deg_count', 'disease', 'diseaseid',
       'tissue', 'data_source', 'library_strategy', 'organism',
       'control_case_sample_count', 'definition', 'n_control', 'n_cases',
       'celltype', 'ids', 'dataset_id', 'combination'],
      dtype='object')
INFO:root:Number of diseases with 3+ datasets: 82
/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/sklearn/model_selection/_split.py:994: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(
INFO:root:Nº of diseases in train split 1: 82
INFO:root:Nº of diseases in test split 1: 38
INFO:root:Nº of datasets in train split 1: 523
INFO:root:Nº of datasets in test split 1: 58
INFO:root:Nº of samples in train split 1: 897
INFO:root:Nº of samples in test split 1: 102
INFO:root:Nº of diseases in train split 2: 82
INFO:root:Nº of diseases in test split 2: 40
INFO:root:Nº of datasets in 

In [ ]:
df_split

,dsaid,accession,platform,deg_count,disease,diseaseid,tissue,data_source,library_strategy,organism,...,test_split_1,test_split_2,test_split_3,test_split_4,test_split_5,test_split_6,test_split_7,test_split_8,test_split_9,test_split_10
5,DSA00006,GSE126342,GPL11154,1000,Myotonic Dystrophy Type 1,C0027126,Skeletal muscle,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,1,0,0,0
14,DSA00015,GSE224056,GPL24676,1000,Gastric Cancer,C0699791,Stomach,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,0,0,1,0
15,DSA00016,GSE224056,GPL24676,112,Gastric Cancer,C0699791,Stomach,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,0,0,1,0
29,DSA00030,GSE215424,GPL24676,1000,Amyotrophic Lateral Sclerosis,C0002736,Muscle,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,1,0,0,0,0
44,DSA00045,GSE223405,GPL16791,1000,Prostate Cancer,C0600139,NaN,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10101,DSA10102,GSE163908,GPL24676,935,Idiopathic Pulmonary Fibrosis,C1800706,NaN,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,0,0,0,1
10120,DSA10121,GSE29006,GPL10999,0,Lung Cancer,C0684249,Lung,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,1,0,0,0
10130,DSA10131,GSE164264,GPL23227,1000,Acquired Immunodeficiency Syndrome,C0001175,Blood,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,1,0,0,0,0
10131,DSA10132,GSE164264,GPL23227,1000,Acquired Immunodeficiency Syndrome,C0001175,Blood,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
df_split_2

,dsaid,accession,platform,deg_count,disease,diseaseid,tissue,data_source,library_strategy,organism,control_case_sample_count,definition,n_control,n_cases,celltype,ids,dataset_id,test_split
5,DSA00006,GSE126342,GPL11154,1000,Myotonic Dystrophy Type 1,C0027126,Skeletal muscle,GEO,RNA-Seq,Homo sapiens,6|16,DO:A myotonic disease that is characterized by...,6,16,C0027126,DSA00006,GSE126342,0
14,DSA00015,GSE224056,GPL24676,1000,Gastric Cancer,C0699791,Stomach,GEO,RNA-Seq,Homo sapiens,4|5,DO:A gastrointestinal system cancer that is lo...,4,5,C0699791,DSA00015,GSE224056,0
15,DSA00016,GSE224056,GPL24676,112,Gastric Cancer,C0699791,Stomach,GEO,RNA-Seq,Homo sapiens,5|5,DO:A gastrointestinal system cancer that is lo...,5,5,C0699791,DSA00016,GSE224056,0
29,DSA00030,GSE215424,GPL24676,1000,Amyotrophic Lateral Sclerosis,C0002736,Muscle,GEO,RNA-Seq,Homo sapiens,5|5,DO:A motor neuron disease that is characterize...,5,5,C0002736,DSA00030,GSE215424,0
44,DSA00045,GSE223405,GPL16791,1000,Prostate Cancer,C0600139,NaN,GEO,RNA-Seq,Homo sapiens,3|3,DO:A prostate cancer that has_material_basis_i...,3,3,C0600139,DSA00045,GSE223405,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10101,DSA10102,GSE163908,GPL24676,935,Idiopathic Pulmonary Fibrosis,C1800706,NaN,GEO,RNA-Seq,Homo sapiens,3|3,DO:A pulmonary fibrosis that is characterized ...,3,3,C1800706,DSA10102,GSE163908,0
10120,DSA10121,GSE29006,GPL10999,0,Lung Cancer,C0684249,Lung,GEO,RNA-Seq,Homo sapiens,2|2,DO:A respiratory system cancer that is located...,2,2,C0684249,DSA10121,GSE29006,0
10130,DSA10131,GSE164264,GPL23227,1000,Acquired Immunodeficiency Syndrome,C0001175,Blood,GEO,RNA-Seq,Homo sapiens,2|2,DO:A viral infectious disease that results in ...,2,2,C0001175,DSA10131,GSE164264,0
10131,DSA10132,GSE164264,GPL23227,1000,Acquired Immunodeficiency Syndrome,C0001175,Blood,GEO,RNA-Seq,Homo sapiens,2|2,DO:A viral infectious disease that results in ...,2,2,C0001175,DSA10132,GSE164264,0


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "A": [1, 2, 1, 3],
    "B": [1, 1, 3, 4],
    "value": [10, 20, 30, 40]
})

# combinations to mask
combos = [(1, 1), (2, 1), (1, 3)]

mask = df[["A", "B"]].apply(tuple, axis=1).isin(combos)

print(mask)
print(df[mask])


0     True
1     True
2     True
3    False
dtype: bool
   A  B  value
0  1  1     10
1  2  1     20
2  1  3     30


In [ ]:
list(zip(df["A"],df["B"]))

[(1, 1), (2, 1), (1, 3), (3, 4)]

In [ ]:
for i in range(10):
    print(f"Nº of diseasses in train split {i+1}: {df_split[df_split[f'test_split_{i+1}'] == 1]['celltype'].nunique()}")
    print(f"Nº of diseasses in train split {i+1}: {df_split[df_split[f'test_split_{i+1}'] == 1]['accession'].nunique()}")

Nº of diseasses in train split 1: 38
Nº of diseasses in train split 1: 58
Nº of diseasses in train split 2: 40
Nº of diseasses in train split 2: 58
Nº of diseasses in train split 3: 38
Nº of diseasses in train split 3: 59
Nº of diseasses in train split 4: 35
Nº of diseasses in train split 4: 59
Nº of diseasses in train split 5: 38
Nº of diseasses in train split 5: 58
Nº of diseasses in train split 6: 34
Nº of diseasses in train split 6: 57
Nº of diseasses in train split 7: 38
Nº of diseasses in train split 7: 58
Nº of diseasses in train split 8: 36
Nº of diseasses in train split 8: 59
Nº of diseasses in train split 9: 39
Nº of diseasses in train split 9: 58
Nº of diseasses in train split 10: 42
Nº of diseasses in train split 10: 57


In [ ]:
df_split_2

In [ ]:
print(f"Nº of diseasses in train split {i+1}: {df_split_2[df_split_2[f'test_split'] == 1]['celltype'].nunique()}")
print(f"Nº of datasets in train split {i+1}: {df_split_2[df_split_2[f'test_split'] == 0]['accession'].nunique()}")

Nº of diseasses in train split 10: 82
Nº of datasets in train split 10: 496


In [ ]:
df_split[df_split["diseaseid"]=="C0002874"]

,dsaid,accession,platform,deg_count,disease,diseaseid,tissue,data_source,library_strategy,organism,...,test_split_1,test_split_2,test_split_3,test_split_4,test_split_5,test_split_6,test_split_7,test_split_8,test_split_9,test_split_10
2634,DSA02635,GSE165870,GPL24676,129,Aplastic Anemia,C0002874,Bone marrow,GEO,RNA-Seq,Homo sapiens,...,0,1,0,0,0,0,0,0,0,0
3216,DSA03217,GSE140844,GPL11154,93,Aplastic Anemia,C0002874,NaN,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,0,0,1,0
9330,DSA09331,GSE145531,GPL20301,1000,Aplastic Anemia,C0002874,Bone marrow,GEO,RNA-Seq,Homo sapiens,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
df_filtered_3.groupby('diseaseid')['accession'].nunique()

diseaseid
C0001175     3
C0002395    29
C0002736    17
C0002874     3
C0003864     3
            ..
C1800706    18
C1839259     3
C1879321     7
C2239176     4
C2936739     3
Name: accession, Length: 82, dtype: int64

In [ ]:
importlib.reload(pp)

<module 'src.preprocessing.pipeline' from '/aloy/home/ddalton/projects/scGPT_playground/src/preprocessing/pipeline.py'>

In [ ]:
dsaids_interest = ['DSA00004',
 'DSA00006',
 'DSA00015',
 'DSA00016',
 'DSA00030',
 'DSA00045',
 'DSA00046',
 'DSA00047',
 'DSA00048',
 'DSA00071',
 'DSA00072',
 'DSA00073',
 'DSA00074',
 'DSA00105',
 'DSA00111',
 'DSA00112',
 'DSA00113',]
df_1 = pp.get_processed_exp_prof_2(dsaids_interest, normalize="log2CPM")


100%|██████████| 17/17 [00:00<00:00, 48.64it/s]


In [ ]:
df_1

In [5]:
df_2 = pp.get_processed_exp_prof(dsaids_interest[:4], normalize="log2CPM")

Processing: 100%|██████████| 4/4 [00:00<00:00, 16.36it/s]


In [4]:
dsaids_interest

['DSA00004',
 'DSA00006',
 'DSA00015',
 'DSA00016',
 'DSA00030',
 'DSA00045',
 'DSA00046',
 'DSA00047',
 'DSA00048',
 'DSA00068',
 'DSA00069',
 'DSA00071',
 'DSA00072',
 'DSA00073',
 'DSA00074',
 'DSA00104',
 'DSA00105',
 'DSA00111',
 'DSA00112',
 'DSA00113',
 'DSA00115',
 'DSA00116',
 'DSA00117',
 'DSA00118',
 'DSA00131',
 'DSA00132',
 'DSA00135',
 'DSA00147',
 'DSA00148',
 'DSA00151',
 'DSA00175',
 'DSA00176',
 'DSA00180',
 'DSA00181',
 'DSA00187',
 'DSA00188',
 'DSA00197',
 'DSA00205',
 'DSA00215',
 'DSA00216',
 'DSA00217',
 'DSA00219',
 'DSA00220',
 'DSA00221',
 'DSA00244',
 'DSA00247',
 'DSA00248',
 'DSA00249',
 'DSA00250',
 'DSA00252',
 'DSA00253',
 'DSA00275',
 'DSA00276',
 'DSA00277',
 'DSA00278',
 'DSA00279',
 'DSA00282',
 'DSA00283',
 'DSA00288',
 'DSA00295',
 'DSA00312',
 'DSA00313',
 'DSA00342',
 'DSA00343',
 'DSA00353',
 'DSA00360',
 'DSA00365',
 'DSA00366',
 'DSA00367',
 'DSA00368',
 'DSA00395',
 'DSA00413',
 'DSA00414',
 'DSA00415',
 'DSA00416',
 'DSA00417',
 'DSA00418',

In [ ]:
df_2